In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)

In [2]:
#Load dataset
df=pd.read_csv("../data/aero_piston_engine_simulation_v1_1.csv")

In [3]:
df.head()

,timestamp_s,engine_id,altitude_m,ambient_temp_C,pressure_kPa,air_density_kg_m3,throttle,load,rpm,air_mass_flow_kg_s,...,torque_Nm,power_W,cht_C,egt_C,oil_temperature_C,oil_pressure_bar,vibration_rms,health_index,degradation,rul_hours
0,0.0,ENG_0042,0.000000,40.000000,101.325000,1.127215,0.75,0.756004,6110.244824,0.015097,...,20.894324,13390.856666,150.050879,635.868136,99.987309,0.174409,0.471967,1.0,1.602852e-10,9.740278
1,1.0,ENG_0042,0.992063,39.993552,101.313083,1.127105,0.75,0.757095,6125.636206,0.097625,...,135.124594,86599.312030,149.573556,623.921001,99.711270,0.226354,0.458027,1.0,3.264474e-09,9.740000
2,2.0,ENG_0042,1.984127,39.987103,101.301166,1.126996,0.75,0.750265,6126.112707,0.097596,...,135.008122,86524.666686,149.805249,610.513781,100.165196,0.210963,0.464967,1.0,9.077059e-09,9.739722
3,3.0,ENG_0042,2.976190,39.980655,101.289251,1.126887,0.75,0.756060,6117.967925,0.097603,...,135.082815,86572.536524,149.628753,600.256351,100.350770,0.196582,0.446174,1.0,1.748562e-08,9.739444
4,4.0,ENG_0042,3.968254,39.974206,101.277337,1.126777,0.75,0.746883,6123.716271,0.097567,...,134.930157,86474.700039,149.719067,588.556505,100.053638,0.206561,0.471848,1.0,2.837881e-08,9.739167


In [4]:
features = [
    "altitude_m",
    "ambient_temp_C",
    "pressure_kPa",
    "air_density_kg_m3",
    "throttle",
    "load",
    "rpm",
    "air_mass_flow_kg_s",
    "fuel_flow_kg_s",
    "torque_Nm",
    "power_W",
    "cht_C",
    "egt_C",
    "oil_temperature_C",
    "oil_pressure_bar",
    "vibration_rms"
]

In [5]:
X=df[features]
y=df["rul_hours"]

In [6]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)

In [7]:
model=Pipeline([
    ("scaler",StandardScaler()),
    ("regressor",LinearRegression())
])

In [8]:
model.fit(X_train,y_train)

,steps,"[('scaler', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


In [9]:
y_pred=model.predict(X_test)

In [10]:
mae=mean_absolute_error(y_test,y_pred)
rmse=np.sqrt(mean_squared_error(y_test,y_pred))
r2=r2_score(y_test,y_pred)

In [11]:
print("Linear Regression")
print("MAE: ",mae)
print("RMSE: ",rmse)
print("R2: ",r2)

Linear Regression
MAE:  0.05199825556925828
RMSE:  0.06540224982066033
R2:  0.9994606886557938


In [12]:
coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.named_steps["regressor"].coef_
})

coefficients["Absolute"] = (
    coefficients["Coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "Absolute",
    ascending=False
)

print(coefficients)

               Feature  Coefficient   Absolute
3    air_density_kg_m3   -45.345243  45.345243
2         pressure_kPa    36.494655  36.494655
10             power_W    24.502172  24.502172
7   air_mass_flow_kg_s   -22.989646  22.989646
0           altitude_m    -4.569068   4.569068
1       ambient_temp_C     4.569068   4.569068
15       vibration_rms     1.285678   1.285678
9            torque_Nm     0.918235   0.918235
5                 load    -0.471244   0.471244
13   oil_temperature_C     0.291095   0.291095
11               cht_C     0.260351   0.260351
8       fuel_flow_kg_s    -0.205767   0.205767
6                  rpm    -0.090767   0.090767
4             throttle     0.079180   0.079180
12               egt_C    -0.001420   0.001420
14    oil_pressure_bar    -0.000207   0.000207


### RANDOM FOREST

In [13]:
from sklearn.ensemble import RandomForestRegressor

rf=RandomForestRegressor(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=1
)

In [14]:
rf.fit(X_train,y_train)
rf_pred=rf.predict(X_test)

In [15]:
rmae=mean_absolute_error(y_test,rf_pred)
rrmse=np.sqrt(mean_squared_error(y_test,rf_pred))
rr2=r2_score(y_test,rf_pred)

In [16]:
print("Random Forest Regression")
print("MAE: ",rmae)
print("RMSE: ",rrmse)
print("R2: ",rr2)

Random Forest Regression
MAE:  0.023498407345661658
RMSE:  0.03448889384867813
R2:  0.9998500271272898
